In [0]:
-- ╔════════════════════════════════════════════════════════════╗
-- ║  03_Review_Silver_Cleaned                                 ║
-- ║  Silver_Cleaned Review Table — Empty DDL Only             ║
-- ╚════════════════════════════════════════════════════════════╝
--
-- Source : yelp_poc_uc.bronze_final.yelp_review (~6.99M rows)
-- Targets (all in yelp_poc_uc.silver_cleaned):
--   yelp_review  (fact table; 1 row per review)
--
-- ════════════════════════════════════════════════════════════
-- RULES & DESIGN PRINCIPLES
-- ════════════════════════════════════════════════════════════
-- (Complete definitions in 01_Business_Silver_Cleaned)
--
-- R1  DDL-Only / No Population
--     Table is created empty. Data population (type corrections,
--     vote clamping, text cleaning) is handled by a separate
--     loading script.
--
-- R2  No Unnecessary Constraint Removal
--     PK/FK/NOT NULL declared inline in CREATE TABLE.
--
-- R4  Referential Integrity via PK/FK
--     PK on review_id and FK to yelp_business are declared
--     inline. FK to yelp_user is DEFERRED.
--
-- R5  Semantic Data Types
--     Bronze Type       →  Silver Semantic Type  → Columns Affected
--     ────────────────────────────────────────────────────────────
--     DOUBLE            →  INT                   → stars
--     STRING datetime   →  TIMESTAMP             → review_timestamp
--     BIGINT            →  BIGINT (clamped ≥ 0)  → useful, funny, cool
--     STRING            →  STRING (cleaned)      → text
--
-- R6  No Ingestion Timestamps
--     Lineage tracked via TBLPROPERTIES metadata.
--
-- R7  Self-Documenting DDL
--     Every column carries an inline comment.
--
-- DQ Issues Addressed (during population):
--   DQ-REVIEW-001 through DQ-REVIEW-010, DQ-REVIEW-014
--
-- Execution order: Run AFTER 01_Business (FK).
--   FK to yelp_user DEFERRED — enabled in 05_User.
-- ════════════════════════════════════════════════════════════


-- ── 1. Review Table (Silver Cleaned) ────────────────────────
-- Bronze path: yelp_review
--                └── 10 columns: review_id, user_id, business_id,
--                    stars, useful, funny, cool, text, date,
--                    is_latest_review
-- Flat fact table — no nested structures requiring child tables.
-- One row per review. Stars DOUBLE→INT, date STRING→TIMESTAMP,
-- votes clamped ≥ 0, text cleaned (control chars, HTML entities,
-- whitespace normalization). ~6.99M rows when populated.

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_review (
  review_id         STRING    NOT NULL  COMMENT 'Unique 22-char Yelp identifier for the review. Primary key.',
  user_id           STRING    NOT NULL  COMMENT 'Review author''s user ID. FK to yelp_user (deferred to 05_User_Silver_Cleaned). Links each review to the user who wrote it.',
  business_id       STRING    NOT NULL  COMMENT 'FK to yelp_business. Unique 22-char Yelp identifier for the reviewed business. Links each review to the business it describes.',
  stars             INT                 COMMENT 'Integer star rating 1–5 assigned by the reviewer. Source: bronze_final.yelp_review.stars DOUBLE cast to INT during population. All 6,990,280 values are whole numbers — no fractional loss (DQ-REVIEW-001).',
  useful            BIGINT              COMMENT 'Count of "useful" votes on the review. Source: bronze_final.yelp_review.useful BIGINT, clamped to ≥ 0 during population to correct 4 rows with value -1 (DQ-REVIEW-003).',
  funny             BIGINT              COMMENT 'Count of "funny" votes on the review. Source: bronze_final.yelp_review.funny BIGINT, clamped to ≥ 0 during population to correct 2 rows with value -1 (DQ-REVIEW-003).',
  cool              BIGINT              COMMENT 'Count of "cool" votes on the review. Source: bronze_final.yelp_review.cool BIGINT, clamped to ≥ 0 during population to correct 1 row with value -1 (DQ-REVIEW-003).',
  review_timestamp  TIMESTAMP           COMMENT 'When the review was posted. Source: bronze_final.yelp_review.date STRING in "YYYY-MM-DD HH:MM:SS" format, cast to TIMESTAMP during population. All 6,990,280 values parse successfully — no null casts (DQ-REVIEW-002).',
  text              STRING              COMMENT 'Full review text body. Source: bronze_final.yelp_review.text STRING, cleaned during population: consecutive spaces collapsed (DQ-REVIEW-007, 34.07% of rows), control characters stripped (DQ-REVIEW-008, 18 rows), HTML entities decoded (DQ-REVIEW-009, 97 rows), excessive newlines collapsed and carriage returns normalized (DQ-REVIEW-010).',
  is_latest_review  BOOLEAN   NOT NULL  COMMENT 'TRUE if this is the user''s most recent review for the business (latest by review_timestamp, ties broken by review_id DESC). Derived via ROW_NUMBER() OVER (PARTITION BY user_id, business_id ORDER BY review_timestamp DESC, review_id DESC). SCD pattern: users revise reviews over time; 89.3% of multi-review pairs have exactly 2 versions, median gap 160 days, 41.7% changed star rating (DQ-REVIEW-014).',
  CONSTRAINT pk_sc_review PRIMARY KEY (review_id),
  CONSTRAINT fk_sc_review_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id),
  -- FK to yelp_user DEFERRED — enabled in 05_User_Silver_Cleaned
  CONSTRAINT fk_sc_review_user
    FOREIGN KEY (user_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_user (user_id)
)
USING DELTA
COMMENT 'Silver_Cleaned review table (DDL-only, no data). Stars DOUBLE→INT, date STRING→TIMESTAMP, votes clamped ≥0, text cleaned. PK on review_id. FK to yelp_business; deferred FK to yelp_user. ~6.99M rows when populated.'
TBLPROPERTIES (
  'quality'                = 'silver_cleaned',
  'pipelines.sourceTable'  = 'yelp_poc_uc.bronze_final.yelp_review',
  'design.populationMode'  = 'ddl_only',
  'dq.issues.resolved'     = 'DQ-REVIEW-001, DQ-REVIEW-002, DQ-REVIEW-003, DQ-REVIEW-007, DQ-REVIEW-008, DQ-REVIEW-009, DQ-REVIEW-010, DQ-REVIEW-014'
);


-- ── 2. Verification ─────────────────────────────────────────

SELECT 'yelp_review' AS table_name, 'silver_cleaned' AS layer, COUNT(*) AS total_rows
FROM yelp_poc_uc.silver_cleaned.yelp_review;